In [21]:
import arcpy
import os
import pandas as pd

# ================= 參數設定區 =================
survey_1 = "OL_P1_2204_corals"
survey_2 = "OL_P1_2209_cor_SimplifyPolyg"

name_1 = "OLP1"
date_1 = "2204"
name_2 = "OLP1"
date_2 = "2209"

output_folder = r"D:\ArcGIS\Morpho_Spatial_Demography\data"

# ================= 空間分析函式 =================

def get_field_mappings(target_layer, join_layer):
    field_mappings = arcpy.FieldMappings()
    desc_t = arcpy.Describe(target_layer)
    desc_j = arcpy.Describe(join_layer)
    
    fields_t = [f.name for f in desc_t.fields if f.type not in ["OID", "Geometry"]]
    fields_j = [f.name for f in desc_j.fields if f.type not in ["OID", "Geometry"]]
    
    common_fields = set(fields_t).intersection(set(fields_j))

    for field_name in fields_t:
        fm = arcpy.FieldMap()
        fm.addInputField(target_layer, field_name)
        new_field = fm.outputField
        if field_name in common_fields:
            new_field.name = f"{field_name}_1"
        fm.outputField = new_field
        field_mappings.addFieldMap(fm)

    for field_name in fields_j:
        fm = arcpy.FieldMap()
        fm.addInputField(join_layer, field_name)
        new_field = fm.outputField
        if field_name in common_fields:
            new_field.name = f"{field_name}_2"
        fm.outputField = new_field
        field_mappings.addFieldMap(fm)
        
    return field_mappings

def perform_spatial_join_and_export(target_layer, join_layer, t_name, t_date, j_date, out_path):
    arcpy.env.overwriteOutput = True
    csv_filename = f"{t_name}_{t_date}SJ{j_date}.csv"
    full_out_path = os.path.join(out_path, csv_filename)
    
    temp_layer_name = f"temp_sj_{t_date}_{j_date}"
    temp_output = f"memory\\{temp_layer_name}"
    
    fms = get_field_mappings(target_layer, join_layer)
    
    arcpy.analysis.SpatialJoin(
        target_features=target_layer,
        join_features=join_layer,
        out_feature_class=temp_output,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        field_mapping=fms,
        match_option="INTERSECT"
    )
    
    arcpy.conversion.ExportTable(temp_output, full_out_path)
    
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    for m in aprx.listMaps():
        for lyr in m.listLayers(temp_layer_name):
            m.removeLayer(lyr)
    arcpy.management.Delete(temp_output)
    
    return full_out_path

# ================= 命運計算與統計邏輯 =================

def calculate_fates(csv_input, out_folder, n1, d1):
    df = pd.read_csv(csv_input)
    
    def fate_logic(row):
        join_count = row['Join_Count']
        id_1, id_2 = row.get('Form_ID_1'), row.get('Form_ID_2')
        area_1, area_2 = row.get('Shape_Area_1'), row.get('Shape_Area_2')

        if join_count == 0:
            return "D"
        elif join_count == 1:
            if id_1 == id_2:
                return "S" if area_1 >= area_2 else "G"
            else:
                return "D" # Fate=Other 且 Join_Count=1 時存入 D
        return "Other"

    # 執行判定
    df['Fate'] = df.apply(fate_logic, axis=1)
    
    # --- 新增：統計部分 ---
    summary = df['Fate'].value_counts().reset_index()
    summary.columns = ['Fate', 'Count']
    summary['Percentage (%)'] = (summary['Count'] / len(df) * 100).round(2)
    
    # 輸出 CSV
    fate_csv_name = f"{n1}{d1}Fates.csv"
    fate_csv_path = os.path.join(out_folder, fate_csv_name)
    df.to_csv(fate_csv_path, index=False, encoding='utf-8-sig')
    
    return fate_csv_path, summary

# ================= 主程式執行 =================

if __name__ == "__main__":
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    print(">>> 步驟 1: 執行 Spatial Join...")
    p1 = perform_spatial_join_and_export(survey_1, survey_2, name_1, date_1, date_2, output_folder)
    
    print(">>> 步驟 2: 計算命運欄位並統計...")
    fate_file, stats = calculate_fates(p1, output_folder, name_1, date_1)

    # 輸出美化統計結果
    print("\n" + "="*50)
    print(f"【命運判定統計結果 - {name_1}_{date_1}】")
    print("-" * 50)
    print(stats.to_string(index=False))
    print("-" * 50)
    print(f"總樣本數: {stats['Count'].sum()}")
    print(f"結果檔案: {os.path.basename(fate_file)}")
    print("="*50)

>>> 步驟 1: 執行 Spatial Join...
>>> 步驟 2: 計算命運欄位並統計...

【命運判定統計結果 - OLP1_2204】
--------------------------------------------------
 Fate  Count  Percentage (%)
    D    632           41.58
    G    435           28.62
Other    366           24.08
    S     87            5.72
--------------------------------------------------
總樣本數: 1520
結果檔案: OLP12204Fates.csv


In [29]:
## 須先完成上一部運算，加入判定Fi 與遺漏的G S D

import pandas as pd

# 1. 設定檔案路徑
csv1_path = r"D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv" #T1命運csv
csv2_path = r"D:\ArcGIS\Morpho_Spatial_Demography\data\OLP1_2209SJ2204.csv" #T2 spatial join T1 csv

try:
    # 2. 讀取資料
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)

    # 初始化 Ramet 欄位 (若不存在則建立)
    if 'Ramet' not in df1.columns:
        df1['Ramet'] = ""

    # 預處理 df2
    df2['Temp_ID'] = df2['Temp_ID'].astype(str).str.strip()
    # 建立查詢表，方便查找拆分後 ID 的 Form 數值
    lookup_table = df2.drop_duplicates(subset=['Temp_ID']).set_index('Temp_ID')

    # 找出 Fate 為 'Other' 的索引
    target_indices = df1[df1['Fate'] == 'Other'].index

    # 3. 核心處理邏輯 (不印出處理過程)
    for idx in target_indices:
        current_id = df1.at[idx, 'ID']
        matched_rows = df2[df2['ID'] == current_id]
        
        true_split_ids = []
        
        if not matched_rows.empty:
            for _, row in matched_rows.iterrows():
                # 拆分 Temp_ID，例如 "190-192" -> ["190", "192"]
                split_parts = str(row['Temp_ID']).split('-')
                for part in split_parts:
                    part = part.strip()
                    if part in lookup_table.index:
                        # 比較該 ID 的 Form_1 是否等於 Form_2
                        if lookup_table.loc[part, 'Form_1'] == lookup_table.loc[part, 'Form_2']:
                            true_split_ids.append(part)
            
            # 依據相同數量更新 Fate 與 Ramet
            same_count = len(true_split_ids)
            if same_count == 0:
                df1.at[idx, 'Fate'] = "D"
            elif same_count == 1:
                df1.at[idx, 'Fate'] = "GorS"
            else: # same_count > 1
                df1.at[idx, 'Fate'] = "Fi"
                df1.at[idx, 'Ramet'] = "-".join(true_split_ids)

    # 4. 存回原始檔案
    df1.to_csv(csv1_path, index=False, encoding='utf-8-sig')
    print(f"處理完成，結果已更新至：{csv1_path}\n")

    # 5. 統計更新後的 Fate 數量與比例
    print("="*30)
    print("最終 Fate 統計結果")
    print("="*30)
    
    counts = df1['Fate'].value_counts()
    percentages = df1['Fate'].value_counts(normalize=True) * 100
    
    # 合併成一個 DataFrame 顯示比較美觀
    stats_df = pd.DataFrame({
        '數量 (Count)': counts,
        '比例 (Percentage %)': percentages.map('{:.2f}%'.format)
    })
    
    print(stats_df)

except Exception as e:
    print(f"執行過程中發生錯誤: {e}")

處理完成，結果已更新至：D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv

最終 Fate 統計結果
       數量 (Count) 比例 (Percentage %)
Fate                               
D             696            45.79%
G             435            28.62%
GorS          134             8.82%
Other          92             6.05%
S              87             5.72%
Fi             76             5.00%


In [47]:
##判定fu

import pandas as pd

# 1. 設定檔案路徑
csv_t1_path = r"D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv"

try:
    # 2. 讀取 CSV
    df = pd.read_csv(csv_t1_path)
    
    # 初始化新欄位 (若不存在)
    for col in ['Fu_T2_ID', 'Fu_group']:
        if col not in df.columns:
            df[col] = ""

    # 定義清理函式：確保轉為字串後沒有 .0
    def clean_to_str(series):
        return pd.to_numeric(series, errors='coerce').fillna(0).astype(int).astype(str)

    # 3. 建立邏輯判斷用的臨時表
    temp_df = df.copy()
    temp_df['ID_str'] = clean_to_str(temp_df['ID'])
    temp_df['Temp_ID_str'] = clean_to_str(temp_df['Temp_ID'])
    temp_df['F1_str'] = clean_to_str(temp_df['Form_1'])
    temp_df['F2_str'] = clean_to_str(temp_df['Form_2'])
    temp_df['Fate_str'] = temp_df['Fate'].astype(str).str.strip()

    # 4. 複合條件過濾
    # 條件 1: Form_1 等於 Form_2
    cond_form = (temp_df['F1_str'] == temp_df['F2_str'])
    # 條件 2: Fate 排除 "Fi" 與 "GorS"
    cond_fate_excluded = ~temp_df['Fate_str'].isin(["Fi", "GorS"])
    # 排除無效 Temp_ID (0 或 空值)
    cond_valid = (temp_df['Temp_ID_str'] != "0") & (temp_df['Temp_ID_str'] != "")
    
    # 初步篩選符合條件的資料
    match_df = temp_df[cond_form & cond_fate_excluded & cond_valid].copy()

    # 條件 3: 排除孤立項 (Temp_ID 必須重複出現)
    mask_duplicated = match_df.duplicated(subset=['Temp_ID_str'], keep=False)
    final_targets = match_df[mask_duplicated].copy()

    update_count = len(final_targets)

    if update_count > 0:
        # 5. 核心邏輯：計算 ID 結合字串 (ID1+ID2)
        # 針對篩選出的資料，依 Temp_ID 分組並用 "+" 串聯 ID
        final_targets['combined_ids'] = final_targets.groupby('Temp_ID_str')['ID_str'].transform(lambda x: "+".join(sorted(x.unique())))

        # 6. 更新原始資料表 df
        target_indices = final_targets.index
        
        # 更新 Fate
        df.loc[target_indices, 'Fate'] = "Fu"
        
        # 更新 Fu_T2_ID (直接存入對應的 Temp_ID 字串)
        df.loc[target_indices, 'Fu_T2_ID'] = final_targets['Temp_ID_str']
        
        # 更新 Fu_group (存入結合後的 ID 字串)
        df.loc[target_indices, 'Fu_group'] = final_targets['combined_ids']
        
        # 7. 存回原始檔案
        df.to_csv(csv_t1_path, index=False, encoding='utf-8-sig')
        print(f"成功更新！共處理 {update_count} 筆資料並標記為 'Fu'。")
        print(f"檔案已儲存至：{csv_t1_path}")
    else:
        print("未發現符合過濾條件的資料，檔案未做變動。")

    # 8. 統計 Fate 數量與比例
    print("\n" + "="*45)
    print(f"{'Fate 類別':<12} | {'數量':<8} | {'比例':<10}")
    print("-" * 45)
    
    counts = df['Fate'].value_counts()
    percentages = df['Fate'].value_counts(normalize=True) * 100
    
    for fate, count in counts.items():
        print(f"{str(fate):<14} | {count:<10} | {percentages[fate]:.2f}%")
    print("="*45)

except Exception as e:
    print(f"執行出錯: {e}")

成功更新！共處理 191 筆資料並標記為 'Fu'。
檔案已儲存至：D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv

Fate 類別      | 數量       | 比例        
---------------------------------------------
D              | 690        | 45.39%
G              | 273        | 17.96%
Fu             | 191        | 12.57%
GorS           | 134        | 8.82%
S              | 79         | 5.20%
Other          | 77         | 5.07%
Fi             | 76         | 5.00%


In [15]:
import arcpy

# 1. 設定參數
csv_path = r"D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv"
layer_name = "OL_P1_2204_corals"
id_field = "ID"
fields_to_update = ["Fate", "Ramet", "Fu_T2_ID", "Fu_group", "Temp_ID"]

# 定義欄位類型對照表
# 除了 Fu_T2_ID 與 Temp_ID 是 LONG，其餘皆為 TEXT
field_type_map = {
    "Fate": "TEXT",
    "Ramet": "TEXT",
    "Fu_T2_ID": "LONG",
    "Fu_group": "TEXT",
    "Temp_ID": "LONG"
}

# 2. 將 CSV 資料讀入字典
# 結構: { ID: (Fate, Ramet, Fu_T2_ID, Fu_group, Temp_ID) }
data_dict = {}
csv_all_fields = [id_field] + fields_to_update

print(f"正在從 {csv_path} 讀取資料...")
try:
    with arcpy.da.SearchCursor(csv_path, csv_all_fields) as cursor:
        for row in cursor:
            # 以 ID 作為 Key，其餘欄位作為 Value
            data_dict[row[0]] = row[1:]
    print(f"讀取完成，共計 {len(data_dict)} 筆紀錄。")
except Exception as e:
    print(f"讀取 CSV 失敗: {e}")

# 3. 檢查並新增圖層欄位
print(f"檢查圖層 {layer_name} 的欄位狀態...")
existing_fields = [f.name for f in arcpy.ListFields(layer_name)]

for field in fields_to_update:
    if field not in existing_fields:
        f_type = field_type_map.get(field, "TEXT")
        arcpy.management.AddField(layer_name, field, f_type)
        print(f"已新增欄位: {field} ({f_type})")
    else:
        print(f"欄位 {field} 已存在，跳過。")

# 4. 執行更新
print("開始寫入資料至圖層...")
update_fields = [id_field] + fields_to_update

with arcpy.da.UpdateCursor(layer_name, update_fields) as cursor:
    count = 0
    for row in cursor:
        current_id = row[0]
        if current_id in data_dict:
            # 將字典中的對應值寫入該行
            row[1:] = data_dict[current_id]
            cursor.updateRow(row)
            count += 1

print(f"成功！已更新 {count} 筆圖徵資料。")

正在從 D:\ArcGIS\Morpho_Spatial_Demography\data\OLP12204Fates.csv 讀取資料...
讀取完成，共計 1520 筆紀錄。
檢查圖層 OL_P1_2204_corals 的欄位狀態...
欄位 Fate 已存在，跳過。
欄位 Ramet 已存在，跳過。
欄位 Fu_T2_ID 已存在，跳過。
欄位 Fu_group 已存在，跳過。
欄位 Temp_ID 已存在，跳過。
開始寫入資料至圖層...
成功！已更新 1520 筆圖徵資料。


In [16]:
##將文字欄位設定成domain

import arcpy

# === 要改的參數 ===
fc = r"D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\OL_P1_2204_corals"
field = "Fate"
domain_name = "Fate_check"

# === 將欄位設定為使用該 domain ===
arcpy.AssignDomainToField_management(fc, field, domain_name)
print(f"已將欄位 {field} 設定為使用 domain：{domain_name}")

# === 從 geodatabase 讀取 domain 對照 ===
desc = arcpy.Describe(fc)
gdb_path = desc.path
domain_list = arcpy.da.ListDomains(gdb_path)

domain = next((d for d in domain_list if d.name == domain_name), None)
if not domain:
    raise ValueError(f"找不到 domain：{domain_name}")

desc_to_code = {v.lower(): k for k, v in domain.codedValues.items()}

# === 更新欄位值 ===
cnt_ok = cnt_missing = 0
codes = set()

with arcpy.da.UpdateCursor(fc, [field]) as cur:
    for (val,) in cur:
        key = (val or "").strip().lower()
        code = desc_to_code.get(key)
        if code is not None:
            cur.updateRow([code])
            cnt_ok += 1
            codes.add(code)
        else:
            cnt_missing += 1

print(f"✅ 完成：已將文字轉換為 domain 代碼。")
print(f"成功轉換 {cnt_ok} 筆，未匹配 {cnt_missing} 筆。")
print(f"共使用 {len(codes)} 種代碼：{sorted(codes)}")


已將欄位 Fate 設定為使用 domain：Fate_check
✅ 完成：已將文字轉換為 domain 代碼。
成功轉換 1520 筆，未匹配 0 筆。
共使用 7 種代碼：['1', '2', '3', '4', '5', '6', '7']


In [ ]:
#加入T2的recruit計算方式